# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the [FAIR² clinicopathological dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the provided URL.

In [ ]:
# Ensure `mlcroissant` library is installed (disable output to avoid noise)
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Display additional basic metadata
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review the available record sets, their fields, and corresponding `@id` identifiers.

In [ ]:
# List all record sets in the dataset
pp = pprint.PrettyPrinter(indent=2)

record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}")

for rec_set in record_sets:
    print(f"\nRecord Set: {rec_set.name}\n  @id: {rec_set.id}")
    print(f"  Description: {getattr(rec_set, 'description', 'N/A')}")
    fields = list(rec_set.fields)
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        col_str = f" (source column @id: {field.column.id})" if hasattr(field, 'column') and hasattr(field.column, 'id') else ''
        print(f"    - {field.name}: @id {field.id}{col_str} (type: {getattr(field, 'data_type', 'N/A')})")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

**Note:** Replace the variables below with the specific record set and field `@id`s if you want to explore other record sets.

In [ ]:
# Get all record set @ids for extraction
record_set_ids = [rec_set.id for rec_set in metadata.record_sets]

dataframes = {}
for rec_id in record_set_ids:
    # Each .records() returns an iterator of dicts; convert to DataFrame
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded record set '{rec_id}' with shape: {df.shape}")

# For demonstration, select the main tabular record set (assume the first one if only one)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nColumns in main DataFrame ({main_record_set_id}):\n{main_df.columns.tolist()}")
    display(main_df.head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's apply standard EDA steps, such as filtering numeric columns, normalizing, and grouping by a categorical field.

*All fields and grouping columns are referenced by their `@id` as per Croissant schema.*

In [ ]:
# Inspect available fields to choose numeric and group fields
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
if len(numeric_fields) == 0:
    print("No numeric columns found in main record set.")
else:
    print("Numeric fields identified:", numeric_fields)

# Example: Use the first numeric field (if available)
numeric_field_id = numeric_fields[0] if numeric_fields else None

# For group field, use the first non-numeric field as string/categorical field
group_candidates = [col for col in main_df.columns if pd.api.types.is_string_dtype(main_df[col])]
group_field_id = group_candidates[0] if group_candidates else None

if numeric_field_id:
    # Example numeric threshold (e.g., could be Age > 50)
    threshold = main_df[numeric_field_id].quantile(0.75)
    # Filter on numeric field
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field in the filtered data
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nGrouped means by '{group_field_id}':")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize distributions or relationships between key fields acquired above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize histogram or boxplot for the selected numeric field
if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field exists, show grouped boxplots
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}' group")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you:
- Loaded and explored metadata for the FAIR² clinicopathological dataset via its Croissant schema
- Reviewed all record sets and their fields, referencing every entity by its `@id`
- Loaded main records and examined available fields
- Demonstrated standard EDA, including filtering on a numeric `@id` field, normalization, and grouping
- Visualized distributions and groupwise differences

This approach can be generalized to any dataset described by Croissant schemas, using only `@id` references to ensure robust, schema-consistent data handling.